# 📘 Sistemas Baseados em Conhecimento — Aula 6 - Integrando Outputs de IA no Motor de Inferência

---

### 🎯 O que muda hoje

Nas Aulas anteriores, os fatos eram escritos à mão:
```python
# Aulas anteriores — fato fictício
fatos = ["evento_visao(stringing)", "confianca(alta)", "vibracao(normal)"]
```

Hoje esses fatos **vêm de IAs reais**, treinadas na disciplina de Reconhecimento de Padrões
```python
# Aula 6 — fato gerado automaticamente a partir de modelos reais
fatos = construir_fatos(chamar_azure(imagem), chamar_svm(janela_vibracao))
```

> 💡 **O Motor de Inferência não muda. As regras não mudam (muito).**
> O que muda é a **origem dos fatos** — antes "simulados", agora produzidos por modelos reais.

---

### 🗺️ Pipeline completo desta aula

```
 [Imagem] ──▶  Azure Custom Vision API  ──▶  { tagName, probability }
                                                         │
 [Sensor] ──▶  SVM de Vibração (.joblib) ──▶  { classe: defeito/normal }
                                                         │
                                               grounding()
                                                         │
                                        [ fatos simbólicos na WM ]
                                                         │
                                          Motor de Inferência
                                                         │
                                         decisao(GO / INVESTIGAR / NOGO)
                                                + por_que() explicável
```


In [2]:
from dataclasses import dataclass
from typing import List, Set, Tuple

# ============================================================
# DEFINIÇÃO DA REGRA
# ============================================================

@dataclass(frozen=True)
class Rule:
    nome: str
    condicoes: tuple
    acao: str
    prioridade: int
    justificativa: str

    def __post_init__(self):
        object.__setattr__(self, 'condicoes', tuple(self.condicoes))

    def match(self, fatos: Set[str]) -> bool:
        return all(c in fatos for c in self.condicoes)

# ============================================================
# BASE DE CONHECIMENTO — Aula 6 (atualizada)
# ============================================================

BASE_REGRAS: List[Rule] = [

    # --- Camada 1: Interpretação dos sensores ---
    Rule(
        nome='R01_VIBRACAO_ERRO_RISCO_ALTO',
        condicoes=['vibracao(erro)'],
        acao='risco_mecanico(alto)',
        prioridade=95,
        justificativa='Vibração em erro indica anomalia mecânica grave.'
    ),
    Rule(
        nome='R02_SINAL_VISUAL_CONFIAVEL',
        condicoes=['confianca(alta)'],
        acao='sinal_visual_confiavel(sim)',
        prioridade=85,
        justificativa='Confiança alta: predição do Azure é confiável.'
    ),
    Rule(
        nome='R03_VIBRACAO_NORMAL_RISCO_BAIXO',
        condicoes=['vibracao(normal)'],
        acao='risco_mecanico(baixo)',
        prioridade=60,
        justificativa='Vibração normal — sem anomalia mecânica detectada.'
    ),

    # --- Camada 2: Decisões base ---
    Rule(
        nome='R04_NOGO_RISCO_ALTO_CRITICO',
        condicoes=['risco_mecanico(alto)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=99,
        justificativa='Risco mecânico alto em peça crítica: parar imediatamente.'
    ),
    Rule(
        nome='R05_INVESTIGAR_RISCO_ALTO_MEDIA',
        condicoes=['risco_mecanico(alto)', 'criticidade(media)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Risco alto em peça de criticidade média: investigar antes de decidir.'
    ),
    Rule(
        nome='R06_GO_TUDO_NORMAL',
        condicoes=['evento_visao(no_defected)', 'vibracao(normal)'],
        acao='decisao(GO)',
        prioridade=70,
        justificativa='Azure diz sem defeito e vibração normal: peça aprovada.'
    ),

    # --- Regra A: leg_broken (suporte quebrado) ---
    Rule(
        nome='R07_LEG_BROKEN_CONFIRMADO',
        condicoes=['evento_visao(leg_broken)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_quebrado(confirmado)',
        prioridade=92,
        justificativa='Azure detectou fratura com alta confiança: defeito estrutural confirmado.'
    ),
    Rule(
        nome='R08_NOGO_DEFEITO_ESTRUTURAL_CRITICO',
        condicoes=['suporte_quebrado(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Defeito estrutural confirmado em peça crítica: NOGO imediato.'
    ),

    # --- Regra B: conflito de sensores ---
    Rule(
        nome='R09_CONFLITO_VISAO_INCERTA_VIBRACAO_ERRO',
        condicoes=['evento_visao(leg_broken)', 'confianca(baixa)', 'vibracao(erro)'],
        acao='decisao(INVESTIGAR)',
        prioridade=89,
        justificativa='Azure viu fratura com baixa confiança, mas vibração confirma anomalia: investigar.'
    ),

    # --- Regra C: bed_no_stick (sem fixação) ---
    Rule(
        nome='R10_BED_NO_STICK_CONFIRMADO',
        condicoes=['evento_visao(bed_no_stick)', 'sinal_visual_confiavel(sim)'],
        acao='sem_fixacao(confirmado)',
        prioridade=90,
        justificativa='Azure detectou falha de adesão com confiança alta.'
    ),
    Rule(
        nome='R11_NOGO_ADESAO_CONFIRMADA_CRITICIDADE_ALTA',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=97,
        justificativa='Falha de adesão em peça crítica: interromper produção.'
    ),
    Rule(
        nome='R12_AJUSTAR_ADESAO_MEDIA_PRAZO_CURTO',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(media)', 'prazo(curto)'],
        acao='decisao(AJUSTAR)',
        prioridade=86,
        justificativa='Criticidade média e prazo curto: ajustar processo rapidamente.'
    ),

    # --- Regra D: no_support (ausência de suporte) ---
    Rule(
        nome='R13_NO_SUPPORT_CONFIRMADO',
        condicoes=['evento_visao(no_support)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_ausente(confirmado)',
        prioridade=90,
        justificativa='Ausência de suporte estrutural confirmada com alta confiança.'
    ),
    Rule(
        nome='R14_NOGO_NO_SUPPORT_CRITICIDADE_ALTA',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Sem suporte em item crítico: bloqueio imediato (NOGO).'
    ),
    Rule(
        nome='R15_INVESTIGAR_SEM_SUPORTE_MEDIA_PRAZO_LONGO',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(media)', 'prazo(longo)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Com criticidade média e prazo longo, investigar causa raiz.'
    ),

    # --- Regra E: no_bottom (base inexistente) ---
    Rule(
        nome='R16_NO_BOTTOM_CONFIRMADO',
        condicoes=['evento_visao(no_bottom)', 'sinal_visual_confiavel(sim)'],
        acao='base_inexistente(confirmado)',
        prioridade=90,
        justificativa='Fundo ausente detectado com confiança alta.'
    ),
    Rule(
        nome='R17_NOGO_NO_BOTTOM_CRITICIDADE_ALTA',
        condicoes=['base_inexistente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=96,
        justificativa='Ausência de fundo em peça crítica: NOGO.'
    ),
    Rule(
        nome='R18_MONITORAR_NO_BOTTOM_BAIXA_PRAZO_LONGO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(baixa)', 'prazo(longo)'],
        acao='decisao(MONITORAR)',
        prioridade=82,
        justificativa='Baixa criticidade e prazo longo: monitorar e reinspecionar.'
    ),
]

print(f'✅ Base de Conhecimento atualizada: {len(BASE_REGRAS)} regras.')

✅ Base de Conhecimento atualizada: 18 regras.


---
## 1. Setup

**O que fazemos:** Importar bibliotecas e carregar o Motor de Inferência das aulas anteriores.

✅ Execute esta célula uma única vez.

In [3]:
import requests
import joblib
import numpy as np
import pandas as pd
import json
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Set, Tuple

print('Ambiente pronto!')

Ambiente pronto!


---
## 2. Motor de Inferência (Aula 4 e 5 — sem alterações)

Copiamos **exatamente** o mesmo motor das aulas anteriores.

Isso demonstra na prática que evoluir a fonte dos dados **não exige mexer na lógica do sistema**.

📌 Não edite nada aqui.

In [4]:
# ============================================================
# MOTOR DE INFERÊNCIA
# ============================================================

@dataclass(frozen=True)
class Rule:
    nome: str
    condicoes: tuple
    acao: str
    prioridade: int
    justificativa: str

    def __post_init__(self):
        object.__setattr__(self, 'condicoes', tuple(self.condicoes))

    def match(self, fatos: Set[str]) -> bool:
        return all(c in fatos for c in self.condicoes)


class WorkingMemory:
    def __init__(self, fatos_iniciais: List[str]):
        self.fatos: Set[str] = set(fatos_iniciais)
        self.historico: List[Tuple[str, str]] = []

    def adicionar(self, fato: str, origem: str = 'initial'):
        if fato not in self.fatos:
            self.fatos.add(fato)
            self.historico.append((fato, origem))


class InferenceEngine:
    def __init__(self, regras: List[Rule]):
        self.regras = sorted(regras, key=lambda r: r.prioridade, reverse=True)
        self.fired_log: List[Dict] = []

    def executar(self, wm: WorkingMemory, verbose: bool = True) -> Dict:
        regras_disparadas = set()
        ciclo = 0
        while True:
            ciclo += 1
            agenda = [r for r in self.regras
                      if r.match(wm.fatos) and r.nome not in regras_disparadas]
            if not agenda:
                if verbose:
                    print(f'  Ciclo {ciclo}: agenda vazia — motor para.')
                break
            selecionada = agenda[0]
            if verbose:
                print(f"  Ciclo {ciclo}: FIRE '{selecionada.nome}' → '{selecionada.acao}'")
            wm.adicionar(selecionada.acao, origem=selecionada.nome)
            regras_disparadas.add(selecionada.nome)
            self.fired_log.append({
                'ciclo': ciclo,
                'regra': selecionada.nome,
                'acao': selecionada.acao,
                'justificativa': selecionada.justificativa
            })
        decisao = next((f for f in wm.fatos if f.startswith('decisao(')), 'decisao(INDEFINIDA)')
        return {'wm_final': wm.fatos, 'decisao': decisao, 'ciclos': ciclo - 1}

    def por_que(self, fato_alvo: str) -> Dict:
        suporte = [e for e in self.fired_log if e['acao'] == fato_alvo]
        if not suporte:
            return {'fato': fato_alvo, 'origem': 'fato inicial (sensor / grounding)'}
        entry = suporte[0]
        regra = next(r for r in self.regras if r.nome == entry['regra'])
        return {
            'fato': fato_alvo,
            'regra': entry['regra'],
            'justificativa': entry['justificativa'],
            'condicoes_necessarias': [self.por_que(c) for c in regra.condicoes]
        }

print('Motor de Inferência carregado')

Motor de Inferência carregado


---
## 3. Módulo de Visão — Azure Custom Vision

### O que o Azure retorna?

Quando enviamos uma imagem para a API, recebemos uma lista com a probabilidade de cada classe treinada:

```json
{
  "predictions": [
    { "tagName": "leg_broken",   "probability": 0.91 },
    { "tagName": "no_defected",  "probability": 0.05 },
    { "tagName": "no_support",   "probability": 0.03 },
    { "tagName": "bed_no_stick", "probability": 0.01 },
    { "tagName": "no_bottom",    "probability": 0.00 }
  ]
}
```

Nossa função seleciona a **tag com maior probabilidade** — essa é a classe prevista.

In [ ]:
# ============================================================
# CONFIGURAÇÃO — Azure Custom Vision
# ============================================================

AZURE_ENDPOINT = 'https://custom3dnunes-prediction.cognitiveservices.azure.com/customvision/v3.0/Prediction/e8d10b1f-7a63-4a9a-b326-85145106ff5b/classify/iterations/Aula3/image'
AZURE_KEY      = os.environ.get('AZURE_CUSTOM_VISION_KEY', 'COLOQUE_SUA_CHAVE_AQUI')


def chamar_azure(imagem_path: str) -> dict:
    """
    Envia uma imagem para o Azure Custom Vision e retorna a predição principal.

    Headers usados (conforme documentação da API):
      - Prediction-Key : chave de autenticação do projeto
      - Content-Type   : application/octet-stream (arquivo binário)

    Retorna: { 'tag': str, 'probabilidade': float, 'todas': list }
    """
    with open(imagem_path, 'rb') as f:
        imagem_bytes = f.read()

    headers = {
        'Prediction-Key': AZURE_KEY,
        'Content-Type':   'application/octet-stream'
    }

    response = requests.post(AZURE_ENDPOINT, headers=headers, data=imagem_bytes)
    response.raise_for_status()

    predictions = response.json()['predictions']
    melhor = max(predictions, key=lambda p: p['probability'])

    return {
        'tag':           melhor['tagName'],
        'probabilidade': melhor['probability'],
        'todas':         predictions
    }

print('Módulo Azure configurado — função chamar_azure() pronta.')

Módulo Azure configurado — função chamar_azure() pronta.


---
## 4. Módulo de Vibração — SVM (.joblib) + CSV de features

### O que é o `vibration_features.csv`?

É o arquivo que vocês geraram na **Aula 5 de RP** (Feature Engineering).
Vocês pegaram as leituras brutas do sensor ADXL345 e calcularam **27 features** por janela de tempo:

| Grupo | Features | Eixos |
|---|---|---|
| Temporais | RMS, média, desvio padrão, máx, mín, curtose, assimetria | X, Y, Z → 21 features |
| Espectrais (FFT) | média FFT, máximo FFT | X, Y, Z → 6 features |

**Resultado:** 32 janelas × 27 features + 1 coluna `label` (`yes`/`no`)

---

### Como funciona na vida real × como funciona hoje

| | Fábrica real | Aula de hoje |
|---|---|---|
| **Fonte** | Sensor ADXL345 ao vivo durante a impressão | Uma linha do `vibration_features.csv` |
| **Features** | Calculadas em tempo real | Já extraídas e salvas na aula de RP |
| **Resultado no SVM** | Idêntico | Idêntico |

> 💡 O mecanismo é exatamente o mesmo. O que muda é só a origem dos 27 números.
> Hoje sorteamos uma linha do CSV. Numa fábrica real, esses números viriam do sensor ao vivo.

---

### ⬆️ Faça o upload do arquivo agora

Execute a célula abaixo e selecione o arquivo `vibration_features.csv` quando solicitado.


---
### ⬆️ Upload dos arquivos — 4 passos

Vamos subir os arquivos separadamente para deixar claro a origem de cada dado.

| Arquivo | O que é | Origem |
|---|---|---|
| `vibration_features.csv` | 27 features por janela de vibração | Aula 5 de RP |
| `svm_model.joblib` | Classificador de vibração treinado | Aula 6 do prof. Paulo |
| `scaler.joblib` | Régua de normalização do treino | Aula 6 de RP |
| `contexto_pecas.csv` | Criticidade e prazo por peça | Sistema MES (simulado) |

> 💡 **Sobre o `contexto_pecas.csv`:** numa fábrica real, criticidade e prazo viriam do MES
> (Manufacturing Execution System) — o software que gerencia a produção.
> Aqui simulamos com um CSV simples, mas a origem e o mecanismo são os mesmos.


In [6]:

# ============================================================
# UPLOAD A — CSV de features de vibração
# "esse é o dataset que vocês geraram na Aula 5 de RP"
# ============================================================

CSV_VIBRACAO_PATH = r'c:\Users\44057824820\Documents\Aula_6_Vibração_Supervisionado\vibration_features - vibration_features.csv'

df_vibracao  = pd.read_csv(CSV_VIBRACAO_PATH)
feature_cols = [c for c in df_vibracao.columns if c != 'label']

# --- Correção de formato numérico brasileiro ---
# Se os valores foram salvos com '.' como separador de milhar e ',' como decimal,
# converte para float corretamente (ex: '2.340,56' → 2340.56 | '2.340' → 2340.0)
for col in feature_cols:
    if df_vibracao[col].dtype == object:
        df_vibracao[col] = (
            df_vibracao[col]
            .str.replace('.', '', regex=False)   # remove separador de milhar
            .str.replace(',', '.', regex=False)  # converte decimal pt-BR → en-US
            .astype(float)
        )

print(f'✅ CSV carregado!')
print(f'   Caminho             : {CSV_VIBRACAO_PATH}')
print(f'   Janelas de vibração : {len(df_vibracao)}')
print(f'   Features por janela : {len(feature_cols)}')
print(f'   Distribuição label  : {df_vibracao["label"].value_counts().to_dict()}')
print(f'   Tipos das colunas   : {df_vibracao[feature_cols].dtypes.unique().tolist()}')
print()
print('💡 Cada linha = uma janela de vibração da impressora.')
print('   Na fábrica real, essa linha viria do sensor ADXL345 ao vivo.')
df_vibracao[feature_cols].head(3)


✅ CSV carregado!
   Caminho             : c:\Users\44057824820\Documents\Aula_6_Vibração_Supervisionado\vibration_features - vibration_features.csv
   Janelas de vibração : 32
   Features por janela : 27
   Distribuição label  : {'yes': 25, 'no': 7}
   Tipos das colunas   : [dtype('float64')]

💡 Cada linha = uma janela de vibração da impressora.
   Na fábrica real, essa linha viria do sensor ADXL345 ao vivo.


,X_rms,X_mean,X_std,X_max,X_min,X_kurt,X_skew,Y_rms,Y_mean,Y_std,...,Z_max,Z_min,Z_kurt,Z_skew,X_fft_mean,X_fft_max,Y_fft_mean,Y_fft_max,Z_fft_mean,Z_fft_max
0,2.572207e+16,-8.687500e+15,2.421058e+16,0.35,-0.78,9.449604e+15,-9.513177e+15,9.065922e+15,8709375.0,2.517484e+15,...,-5.73,-10.36,2.795516e+15,1.974631e+16,1.839632e+16,5.560000e+16,3.531030e+16,5.574000e+16,2.243877e+15,5.141300e+15
1,2.359654e+16,-7.921875e+06,2.222703e+15,0.39,-0.67,1.861876e+16,-4.431290e+15,9.250591e+15,89.0,2.522585e+16,...,-5.73,-10.40,3.843612e+15,1.044100e+16,1.645735e+16,5.070000e+02,3.537728e+16,5.696000e+16,2.187186e+16,5.101700e+16
2,2.166434e+16,-9.812500e+15,1.931472e+16,0.39,-0.63,7.671535e+15,-2.715419e+15,1.025103e+15,100609375.0,1.964976e+16,...,11.77,-10.40,-1.971454e+16,1.146105e+16,1.524749e+16,6.280000e+02,3.356793e+16,6.439000e+03,3.977807e+13,3.962990e+15


In [7]:

# ============================================================
# UPLOAD B — Modelos treinados na aula de RP
# "esses são os modelos que vocês treinaram e exportaram com joblib"
# svm_model.joblib + scaler.joblib
# ============================================================

SVM_MODEL_PATH = r'c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\svm_model.joblib'
SCALER_PATH    = r'c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\scaler.joblib'

svm_vibracao    = joblib.load(SVM_MODEL_PATH)
scaler_vibracao = joblib.load(SCALER_PATH)

print(f'✅ SVM carregado!')
print(f'   Caminho         : {SVM_MODEL_PATH}')
print(f'   Kernel          : {svm_vibracao.kernel}')
print(f'   Classes         : {list(svm_vibracao.classes_)}')
print(f'   Vetores suporte : {svm_vibracao.n_support_}')
print(f'\n✅ Scaler carregado!')
print(f'   Caminho         : {SCALER_PATH}')
print(f'   Features        : {scaler_vibracao.n_features_in_}')
print()
print('💡 svm_model.joblib  = o cérebro do classificador (o que vocês treinaram)')
print('   scaler.joblib     = a régua de normalização (obrigatória — sem ela, resultado errado)')


✅ SVM carregado!
   Caminho         : c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\svm_model.joblib
   Kernel          : rbf
   Classes         : [np.str_('defected'), np.str_('no_defected')]
   Vetores suporte : [175 181]

✅ Scaler carregado!
   Caminho         : c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\scaler.joblib
   Features        : 7

💡 svm_model.joblib  = o cérebro do classificador (o que vocês treinaram)
   scaler.joblib     = a régua de normalização (obrigatória — sem ela, resultado errado)


In [9]:

# ============================================================
# UPLOAD C — Contexto operacional das peças (simula MES)
# "numa fábrica real isso viria do sistema de gestão da produção"
# ============================================================

CONTEXTO_PATH = r'c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\contexto_pecas - contexto_pecas.csv'

df_contexto = pd.read_csv(CONTEXTO_PATH)
df_contexto = df_contexto.set_index('id')  # id vira índice para busca rápida

print(f'✅ Contexto carregado!')
print(f'   Caminho : {CONTEXTO_PATH}')
print()
print(df_contexto.to_string())
print()
print('💡 Esses dados representam o que o MES saberia sobre cada peça:')
print('   criticidade → qual o impacto se essa peça falhar')
print('   prazo_horas → quanto tempo falta para a entrega')
print()
print('   O ID da peça é o elo que une os três sistemas:')
print('   Azure (visão) + SVM (vibração) + MES (contexto) → mesma peça')


✅ Contexto carregado!
   Caminho : c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\contexto_pecas - contexto_pecas.csv

             criticidade  prazo_horas
id                                   
scratch_0           alta           10
scratch_1          media           36
scratch_2_9         alta            8
scratch_2_10       media           20

💡 Esses dados representam o que o MES saberia sobre cada peça:
   criticidade → qual o impacto se essa peça falhar
   prazo_horas → quanto tempo falta para a entrega

   O ID da peça é o elo que une os três sistemas:
   Azure (visão) + SVM (vibração) + MES (contexto) → mesma peça


In [10]:

# ============================================================
# FUNÇÕES DO MÓDULO DE VIBRAÇÃO
# ============================================================

def sortear_leitura_vibracao(seed: int = None) -> dict:
    """
    Sorteia uma linha do CSV — simula uma leitura ao vivo do sensor ADXL345.
    Em produção: esse dicionário viria do sensor em tempo real.
    """
    linha = df_vibracao[feature_cols].sample(1, random_state=seed).iloc[0]
    return linha.to_dict()


def chamar_svm_vibracao(features: dict) -> dict:
    """
    Classifica uma janela de vibração com o SVM.

    Seleciona automaticamente as features corretas:
      - Se o scaler foi treinado com nomes de colunas (feature_names_in_),
        usa exatamente essas colunas na mesma ordem.
      - Caso contrário, usa as primeiras n_features_in_ features do dicionário.

    Retorna: { 'classe': 'yes' | 'no' }
    """
    n_esperado = scaler_vibracao.n_features_in_

    if hasattr(scaler_vibracao, 'feature_names_in_'):
        # Scaler treinado com DataFrame — sabe os nomes das colunas
        cols = list(scaler_vibracao.feature_names_in_)
        X = np.array([features[c] for c in cols]).reshape(1, -1)
    else:
        # Scaler treinado com array — usa as primeiras n features
        vals = list(features.values())[:n_esperado]
        X    = np.array(vals).reshape(1, -1)

    X_scaled = scaler_vibracao.transform(X)
    classe   = svm_vibracao.predict(X_scaled)[0]
    return {'classe': classe}


# ============================================================
# DEMONSTRAÇÃO — pipeline completo de vibração
# ============================================================

print('=== Pipeline de Vibração — passo a passo ===\n')
print(f'ℹ️  CSV tem {len(feature_cols)} features | Scaler espera {scaler_vibracao.n_features_in_} features')
if hasattr(scaler_vibracao, 'feature_names_in_'):
    print(f'   Features do scaler : {list(scaler_vibracao.feature_names_in_)}')
else:
    print(f'   Usando as primeiras {scaler_vibracao.n_features_in_} colunas do CSV')
print()

leitura   = sortear_leitura_vibracao(seed=42)
resultado = chamar_svm_vibracao(leitura)

print(f'PASSO 1 — Leitura sorteada do CSV (primeiras 4 features):')
print(f'  { {k: round(v,4) for k,v in list(leitura.items())[:4]} } ...')
print()
print(f'PASSO 2 — Normalização com scaler → vetor pronto para o SVM')
print()
print(f'PASSO 3 — SVM classificou como: "{resultado["classe"]}" '
      f'({"⚠️ defeito detectado" if resultado["classe"]=="yes" else "✅ operação normal"})')
print()
print('─' * 55)
print('💡 Na fábrica real: os números viriam do sensor ao vivo.')
print('   O resultado seria o mesmo: apenas "yes" ou "no".')
print('   Só isso interessa para o Motor de Inferência.')


=== Pipeline de Vibração — passo a passo ===

ℹ️  CSV tem 27 features | Scaler espera 7 features
   Usando as primeiras 7 colunas do CSV

PASSO 1 — Leitura sorteada do CSV (primeiras 4 features):
  {'X_rms': 2340205653356130.0, 'X_mean': -168125.0, 'X_std': 1.62787006161425e+16, 'X_max': 0.27} ...

PASSO 2 — Normalização com scaler → vetor pronto para o SVM

PASSO 3 — SVM classificou como: "no_defected" (✅ operação normal)

───────────────────────────────────────────────────────
💡 Na fábrica real: os números viriam do sensor ao vivo.
   O resultado seria o mesmo: apenas "yes" ou "no".
   Só isso interessa para o Motor de Inferência.


---
## 5. Grounding — Da IA ao Símbolo

Este é o **ponto de conexão** entre as duas disciplinas.

O motor não entende `0.91` ou `"yes"`. Ele entende `"confianca(alta)"` e `"vibracao(erro)"` — os mesmos símbolos das Aulas 3–5.

A função de grounding faz essa tradução, usando os limites definidos na Aula 3:

```
OUTPUT DA IA                  GROUNDING               SÍMBOLO (WM)
probability = 0.91   ───────────────────────────▶   confianca(alta)
tagName = leg_broken ───────────────────────────▶   evento_visao(leg_broken)
classe = "yes"       ───────────────────────────▶   vibracao(erro)
classe = "no"        ───────────────────────────▶   vibracao(normal)
criticidade = alta   ───────────────────────────▶   criticidade(alta)
prazo = 10h          ───────────────────────────▶   prazo(curto)
```


In [11]:
# ============================================================
# LIMITES DE GROUNDING — mesmos da Aula 3
# ============================================================

LIMITES = {
    'confianca_alta': 0.80,   # prob >= 0.80 → confianca(alta)
    'prazo_curto':    24,     # horas <= 24  → prazo(curto)
}


def grounding_visao(output_azure: dict) -> List[str]:
    """
    Converte o output do Azure Custom Vision em símbolos para a WM.
    Entrada : { 'tag': 'leg_broken', 'probabilidade': 0.91 }
    Saída   : [ 'evento_visao(leg_broken)', 'confianca(alta)' ]
    """
    simbolos = [f"evento_visao({output_azure['tag']})"]
    if output_azure['probabilidade'] >= LIMITES['confianca_alta']:
        simbolos.append('confianca(alta)')
    else:
        simbolos.append('confianca(baixa)')
    return simbolos


def grounding_vibracao(output_svm: dict) -> List[str]:
    """
    Converte o output do SVM de vibração em símbolo para a WM.
    Entrada : { 'classe': 'yes' } ou { 'classe': 'no' }
    Saída   : [ 'vibracao(erro)' ] ou [ 'vibracao(normal)' ]
    """
    return ['vibracao(erro)'] if output_svm['classe'] == 'yes' else ['vibracao(normal)']


def grounding_contexto(contexto: dict) -> List[str]:
    """
    Converte o contexto operacional em símbolos para a WM.
    Entrada : { 'criticidade': 'alta', 'prazo_horas': 10 }
    Saída   : [ 'criticidade(alta)', 'prazo(curto)' ]
    """
    simbolos = [f"criticidade({contexto['criticidade']})"]
    simbolos.append('prazo(curto)' if contexto['prazo_horas'] <= LIMITES['prazo_curto'] else 'prazo(longo)')
    return simbolos


def construir_fatos(output_azure: dict, output_svm: dict, contexto: dict) -> List[str]:
    """
    Une os três módulos de grounding em uma lista de fatos simbólicos.
    Essa lista é a entrada direta da WorkingMemory do Motor de Inferência.
    """
    return grounding_visao(output_azure) + grounding_vibracao(output_svm) + grounding_contexto(contexto)


# --- Demonstração passo a passo ---
print('=== Grounding passo a passo — P1 ===')
out_azure_ex = {'tag': 'leg_broken', 'probabilidade': 0.91}
out_svm_ex   = {'classe': 'no'}
ctx_ex       = {'criticidade': 'alta', 'prazo_horas': 10}

print(f"  Azure    : tag={out_azure_ex['tag']}, prob={out_azure_ex['probabilidade']}")
print(f"  SVM      : classe={out_svm_ex['classe']}")
print(f"  Contexto : {ctx_ex}")
print()
print(f"  → visão     : {grounding_visao(out_azure_ex)}")
print(f"  → vibração  : {grounding_vibracao(out_svm_ex)}")
print(f"  → contexto  : {grounding_contexto(ctx_ex)}")
print()
print(f"  ✅ WM inicial: {construir_fatos(out_azure_ex, out_svm_ex, ctx_ex)}")

=== Grounding passo a passo — P1 ===
  Azure    : tag=leg_broken, prob=0.91
  SVM      : classe=no
  Contexto : {'criticidade': 'alta', 'prazo_horas': 10}

  → visão     : ['evento_visao(leg_broken)', 'confianca(alta)']
  → vibração  : ['vibracao(normal)']
  → contexto  : ['criticidade(alta)', 'prazo(curto)']

  ✅ WM inicial: ['evento_visao(leg_broken)', 'confianca(alta)', 'vibracao(normal)', 'criticidade(alta)', 'prazo(curto)']


---
## 6. Gerando os Fatos Reais — Chamando os Dois Modelos

Agora conectamos tudo.

Para cada amostra vamos:
1. **Enviar a imagem** para o Azure → receber a classificação visual real
2. **Sortear uma leitura** do CSV de vibração → passar para o SVM → receber `yes/no`
3. **Aplicar o grounding** nos dois resultados + contexto operacional
4. **Montar a lista de fatos** que entra na WorkingMemory do motor

```
imagem_P1.jpg ──▶ Azure API ──▶ { tag: 'leg_broken', prob: 0.91 }
                                          │
linha do CSV  ──▶ SVM       ──▶ { classe: 'no' }         │
                                          │               │
contexto      ──────────────────────────  │               │
                                          ▼               │
                               grounding()                │
                                          │               │
                                          ▼               │
          [ 'evento_visao(leg_broken)', 'confianca(alta)',  ◀──┘
            'vibracao(normal)', 'criticidade(alta)', 'prazo(curto)' ]
```

> ⚠️ **Lembrete:** a imagem e a leitura de vibração não são da mesma peça física hoje —  
> estamos demonstrando o mecanismo. No projeto final, elas devem corresponder à mesma impressão.


In [28]:
# ============================================================
# CARREGAMENTO D — Imagens das peças para o Azure
# (Adaptado para VS Code — usa caminhos locais ao invés de upload)
# ============================================================

import os
from pathlib import Path

# Pasta base onde estão as imagens
ARCHIVE_PATH = Path(r'c:\Users\44057824820\Documents\3° Modulo sistemas baseados em conhecimento\archive')

# Mapeia os IDs do contexto_pecas.csv para as imagens correspondentes
# Os IDs devem corresponder aos do CSV de contexto!
IMAGENS_SELECIONADAS = [
    ('scratch_0', ARCHIVE_PATH / 'defected' / 'leg_broken_0.jpg'),
    ('scratch_1', ARCHIVE_PATH / 'defected' / 'bed_not_stick_0.jpg'),
    ('scratch_2_9', ARCHIVE_PATH / 'defected' / 'no_bottom_0.jpg'),
    ('scratch_2_10', ARCHIVE_PATH / 'defected' / 'no_support_0.jpg'),
]

IMAGENS = []
print('📂 Carregando imagens locais...\n')

for pid, caminho in IMAGENS_SELECIONADAS:
    caminho = Path(caminho)
    if caminho.exists():
        IMAGENS.append({'id': pid, 'imagem': str(caminho)})
        print(f'   ✅ {pid}: {caminho.name}')
    else:
        print(f'   ⚠️  {pid}: arquivo não encontrado - {caminho}')

IMAGENS = sorted(IMAGENS, key=lambda x: x['id'])

print(f'\n{len(IMAGENS)} imagem(ns) prontas para enviar ao Azure.')
print(f'   IDs carregados : {[img["id"] for img in IMAGENS]}')
print(f'   IDs no contexto: {list(df_contexto.index)}')

# Avisa se algum ID não bate com o contexto
ids_sem_contexto = [img['id'] for img in IMAGENS if img['id'] not in df_contexto.index]
if ids_sem_contexto:
    print(f'\n⚠️  IDs sem contexto no CSV: {ids_sem_contexto}')
    print('   Adicione esses IDs ao contexto_pecas.csv ou ajuste os IDs acima.')
else:
    print('\n✅ Todos os IDs encontrados no contexto — pronto para rodar!')

📂 Carregando imagens locais...

   ✅ scratch_0: leg_broken_0.jpg
   ✅ scratch_1: bed_not_stick_0.jpg
   ✅ scratch_2_9: no_bottom_0.jpg
   ✅ scratch_2_10: no_support_0.jpg

4 imagem(ns) prontas para enviar ao Azure.
   IDs carregados : ['scratch_0', 'scratch_1', 'scratch_2_10', 'scratch_2_9']
   IDs no contexto: ['scratch_0', 'scratch_1', 'scratch_2_9', 'scratch_2_10']

✅ Todos os IDs encontrados no contexto — pronto para rodar!


In [3]:
# ============================================================
# GERANDO OS FATOS REAIS — Azure + SVM + Contexto
# ============================================================

FATOS_REAIS    = {}
OUTPUTS_BRUTOS = {}

print('Chamando Azure + SVM e construindo fatos...\n')
print(f'{"ID":<12} {"Azure (tag)":<16} {"Prob":>5}  {"Conf":<6}  {"SVM":>6}  {"Criticidade":<12} {"Prazo"}')
print('-' * 90)

for i, item in enumerate(IMAGENS):
    pid = item['id']

    # Verifica se o ID da imagem existe no contexto (MES)
    if pid not in df_contexto.index:
        print(f'⚠️  ID "{pid}" não encontrado no contexto_pecas.csv — pulando.')
        continue

    # 1. Visão: chamada real à API do Azure
    out_azure = chamar_azure(item['imagem'])

    # 2. Vibração: sorteia linha do CSV → SVM
    #    Em produção: viria do sensor ADXL345 ao vivo
    leitura_vib = sortear_leitura_vibracao(seed=i)
    out_svm     = chamar_svm_vibracao(leitura_vib)

    # 3. Contexto: vem do CSV que simula o MES
    ctx = df_contexto.loc[pid].to_dict()

    # 4. Grounding → fatos simbólicos
    fatos = construir_fatos(out_azure, out_svm, ctx)

    FATOS_REAIS[pid]    = fatos
    OUTPUTS_BRUTOS[pid] = {'azure': out_azure, 'svm': out_svm, 'contexto': ctx}

    conf = 'alta' if out_azure['probabilidade'] >= LIMITES['confianca_alta'] else 'baixa'
    print(f"{pid:<12} {out_azure['tag']:<16} {out_azure['probabilidade']:>5.2f}  {conf:<6}  "
          f"{out_svm['classe']:>6}  {ctx['criticidade']:<12} {ctx['prazo_horas']}h")

print('\n✅ Fatos reais construídos!')
print()
print('Origem de cada dado:')
print('  Azure    → tag + probabilidade   (API real)')
print('  SVM      → yes / no              (modelo real + linha do CSV)')
print('  Contexto → criticidade + prazo   (CSV que simula o MES)')


Chamando Azure + SVM e construindo fatos...

ID           Azure (tag)       Prob  Conf       SVM  Criticidade  Prazo
------------------------------------------------------------------------------------------


NameError: name 'IMAGENS' is not defined

---
## 7. Base de Conhecimento — Regras da Aula 6

Todas as regras foram criadas para os dados reais do Azure + SVM.
Não há herança das aulas anteriores — as classes do Azure (`leg_broken`, `no_defected`, etc.)
são diferentes das classes fictícias usadas até a Aula 5.

---

### Camada 1 — Interpretação dos sensores

Regras genéricas que funcionam independente da classe detectada:
```
SE  vibracao(erro)     → risco_mecanico(alto)
SE  vibracao(normal)   → risco_mecanico(baixo)
SE  confianca(alta)    → sinal_visual_confiavel(sim)
```

### Camada 2 — Decisões base
```
SE  risco_mecanico(alto)  E  criticidade(alta)   → decisao(NOGO)
SE  risco_mecanico(alto)  E  criticidade(media)  → decisao(INVESTIGAR)
SE  evento_visao(no_defected)  E  vibracao(normal) → decisao(GO)
```

---

### 🔎 Regras criadas em aula

**Regra A — Fratura confirmada pelo Azure**
```
SE  evento_visao(leg_broken)  E  sinal_visual_confiavel(sim)
ENTÃO  defeito_estrutural(confirmado)
```
> O Azure detectou fratura com confiança alta — defeito estrutural confirmado.
```
SE  defeito_estrutural(confirmado)  E  criticidade(alta)
ENTÃO  decisao(NOGO)
```
> Defeito estrutural confirmado em peça crítica — parar imediatamente.

**Regra B — Conflito de sensores**
```
SE  evento_visao(leg_broken)  E  confianca(baixa)  E  vibracao(erro)
ENTÃO  decisao(INVESTIGAR)
```
> A câmera hesitou. A vibração confirmou anomalia. Não para — mas não ignora.

---

In [34]:
from dataclasses import dataclass
from typing import List, Set, Tuple

# ============================================================
# DEFINIÇÃO DA REGRA
# ============================================================

@dataclass(frozen=True)
class Rule:
    nome: str
    condicoes: tuple
    acao: str
    prioridade: int
    justificativa: str

    def __post_init__(self):
        object.__setattr__(self, 'condicoes', tuple(self.condicoes))

    def match(self, fatos: Set[str]) -> bool:
        return all(c in fatos for c in self.condicoes)

# ============================================================
# BASE DE CONHECIMENTO — Aula 6 (atualizada)
# ============================================================

BASE_REGRAS: List[Rule] = [

    # --- Camada 1: Interpretação dos sensores ---
    Rule(
        nome='R01_VIBRACAO_ERRO_RISCO_ALTO',
        condicoes=['vibracao(erro)'],
        acao='risco_mecanico(alto)',
        prioridade=95,
        justificativa='Vibração em erro indica anomalia mecânica grave.'
    ),
    Rule(
        nome='R02_SINAL_VISUAL_CONFIAVEL',
        condicoes=['confianca(alta)'],
        acao='sinal_visual_confiavel(sim)',
        prioridade=85,
        justificativa='Confiança alta: predição do Azure é confiável.'
    ),
    Rule(
        nome='R03_VIBRACAO_NORMAL_RISCO_BAIXO',
        condicoes=['vibracao(normal)'],
        acao='risco_mecanico(baixo)',
        prioridade=60,
        justificativa='Vibração normal — sem anomalia mecânica detectada.'
    ),

    # --- Camada 2: Decisões base ---
    Rule(
        nome='R04_NOGO_RISCO_ALTO_CRITICO',
        condicoes=['risco_mecanico(alto)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=99,
        justificativa='Risco mecânico alto em peça crítica: parar imediatamente.'
    ),
    Rule(
        nome='R05_INVESTIGAR_RISCO_ALTO_MEDIA',
        condicoes=['risco_mecanico(alto)', 'criticidade(media)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Risco alto em peça de criticidade média: investigar antes de decidir.'
    ),
    Rule(
        nome='R06_GO_TUDO_NORMAL',
        condicoes=['evento_visao(no_defected)', 'vibracao(normal)'],
        acao='decisao(GO)',
        prioridade=70,
        justificativa='Azure diz sem defeito e vibração normal: peça aprovada.'
    ),

    # --- Regra A: leg_broken (suporte quebrado) ---
    Rule(
        nome='R07_LEG_BROKEN_CONFIRMADO',
        condicoes=['evento_visao(leg_broken)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_quebrado(confirmado)',
        prioridade=92,
        justificativa='Azure detectou fratura com alta confiança: defeito estrutural confirmado.'
    ),
    Rule(
        nome='R08_NOGO_DEFEITO_ESTRUTURAL_CRITICO',
        condicoes=['suporte_quebrado(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Defeito estrutural confirmado em peça crítica: NOGO imediato.'
    ),

    # --- Regra B: conflito de sensores ---
    Rule(
        nome='R09_CONFLITO_VISAO_INCERTA_VIBRACAO_ERRO',
        condicoes=['evento_visao(leg_broken)', 'confianca(baixa)', 'vibracao(erro)'],
        acao='decisao(INVESTIGAR)',
        prioridade=89,
        justificativa='Azure viu fratura com baixa confiança, mas vibração confirma anomalia: investigar.'
    ),

    # --- Regra C: bed_no_stick (sem fixação) ---
    Rule(
        nome='R10_BED_NO_STICK_CONFIRMADO',
        condicoes=['evento_visao(bed_no_stick)', 'sinal_visual_confiavel(sim)'],
        acao='sem_fixacao(confirmado)',
        prioridade=90,
        justificativa='Azure detectou falha de adesão com confiança alta.'
    ),
    Rule(
        nome='R11_NOGO_ADESAO_CONFIRMADA_CRITICIDADE_ALTA',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=97,
        justificativa='Falha de adesão em peça crítica: interromper produção.'
    ),
    Rule(
        nome='R12_AJUSTAR_ADESAO_MEDIA_PRAZO_CURTO',
        condicoes=['sem_fixacao(confirmado)', 'criticidade(media)', 'prazo(curto)'],
        acao='decisao(AJUSTAR)',
        prioridade=86,
        justificativa='Criticidade média e prazo curto: ajustar processo rapidamente.'
    ),

    # --- Regra D: no_support (ausência de suporte) ---
    Rule(
        nome='R13_NO_SUPPORT_CONFIRMADO',
        condicoes=['evento_visao(no_support)', 'sinal_visual_confiavel(sim)'],
        acao='suporte_ausente(confirmado)',
        prioridade=90,
        justificativa='Ausência de suporte estrutural confirmada com alta confiança.'
    ),
    Rule(
        nome='R14_NOGO_NO_SUPPORT_CRITICIDADE_ALTA',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=98,
        justificativa='Sem suporte em item crítico: bloqueio imediato (NOGO).'
    ),
    Rule(
        nome='R15_INVESTIGAR_SEM_SUPORTE_MEDIA_PRAZO_LONGO',
        condicoes=['suporte_ausente(confirmado)', 'criticidade(media)', 'prazo(longo)'],
        acao='decisao(INVESTIGAR)',
        prioridade=88,
        justificativa='Com criticidade média e prazo longo, investigar causa raiz.'
    ),

    # --- Regra E: no_bottom (base inexistente) ---
    Rule(
        nome='R16_NO_BOTTOM_CONFIRMADO',
        condicoes=['evento_visao(no_bottom)', 'sinal_visual_confiavel(sim)'],
        acao='base_inexistente(confirmado)',
        prioridade=90,
        justificativa='Fundo ausente detectado com confiança alta.'
    ),
    Rule(
        nome='R17_NOGO_NO_BOTTOM_CRITICIDADE_ALTA',
        condicoes=['base_inexistente(confirmado)', 'criticidade(alta)'],
        acao='decisao(NOGO)',
        prioridade=96,
        justificativa='Ausência de fundo em peça crítica: NOGO.'
    ),
    Rule(
        nome='R18_MONITORAR_NO_BOTTOM_BAIXA_PRAZO_LONGO',
        condicoes=['base_inexistente(confirmado)', 'criticidade(baixa)', 'prazo(longo)'],
        acao='decisao(MONITORAR)',
        prioridade=82,
        justificativa='Baixa criticidade e prazo longo: monitorar e reinspecionar.'
    ),
]

print(f'✅ Base de Conhecimento atualizada: {len(BASE_REGRAS)} regras.')

✅ Base de Conhecimento atualizada: 18 regras.


---
## 8. Rodando o Motor com Fatos Reais

Chegamos ao momento central: **o motor recebe fatos que vieram de IAs reais**.

Ele não sabe — e não precisa saber — se os fatos vieram de uma API do Azure ou de um `.joblib`. Ele apenas aplica as regras.

> 💡 Esta é a separação entre **percepção** (modelos de ML, sensores) e **raciocínio** (motor, BK).  
> Sistemas especialistas industriais sérios são construídos exatamente assim.

In [2]:
# ============================================================
# HELPERS — mesmos padrões das Aulas 4/5
# ============================================================

def rodar_motor(pid: str, verbose: bool = True) -> Tuple[InferenceEngine, Dict]:
    fatos = FATOS_REAIS[pid]
    wm    = WorkingMemory(fatos)
    eng   = InferenceEngine(BASE_REGRAS)
    if verbose:
        print(f"\n{'='*60}")
        print(f"  CASO {pid} | Fatos iniciais: {sorted(fatos)}")
        print(f"{'='*60}")
    resultado = eng.executar(wm, verbose=verbose)
    return eng, resultado


def resumo(pid: str, resultado: Dict):
    out = OUTPUTS_BRUTOS[pid]
    print(f'\n--- RESUMO {pid} ---')
    print(f"  Azure   : {out['azure']['tag']} (prob={out['azure']['probabilidade']:.2f})")
    print(f"  SVM     : {out['svm']['classe']}")
    print(f"  Decisão : {resultado['decisao']}")
    print(f"  WM final: {sorted(resultado['wm_final'])}")


# ============================================================
# Mostra os IDs disponíveis para teste
# ============================================================
print('IDs disponíveis em FATOS_REAIS:')
print(f'  {list(FATOS_REAIS.keys())}')
print()

# ============================================================
# scratch_0: leg_broken (suporte quebrado)
# ============================================================
if 'scratch_0' in FATOS_REAIS:
    eng_p1, res_p1 = rodar_motor('scratch_0', verbose=True)
    resumo('scratch_0', res_p1)
    print('\nÁrvore por_que():')
    print(json.dumps(eng_p1.por_que(res_p1['decisao']), indent=2, ensure_ascii=False))
else:
    print('⚠️ scratch_0 não encontrado em FATOS_REAIS')

IDs disponíveis em FATOS_REAIS:


NameError: name 'FATOS_REAIS' is not defined

In [1]:
# ============================================================
# scratch_1: bed_no_stick (sem fixação)
# ============================================================
if 'scratch_1' in FATOS_REAIS:
    eng_p2, res_p2 = rodar_motor('scratch_1', verbose=True)
    resumo('scratch_1', res_p2)
    print('\nÁrvore por_que():')
    print(json.dumps(eng_p2.por_que(res_p2['decisao']), indent=2, ensure_ascii=False))
else:
    print('⚠️ scratch_1 não encontrado em FATOS_REAIS')

NameError: name 'FATOS_REAIS' is not defined

---
## 9. Tabela Comparativa — Todos os Casos

In [37]:
linhas = []
for item in IMAGENS:
    pid = item['id']
    _, res = rodar_motor(pid, verbose=False)
    out = OUTPUTS_BRUTOS[pid]
    ctx = df_contexto.loc[pid].to_dict()
    linhas.append({
        'ID':          pid,
        'Azure (tag)': out['azure']['tag'],
        'Confiança':   'alta' if out['azure']['probabilidade'] >= LIMITES['confianca_alta'] else 'baixa',
        'Vibração':    out['svm']['classe'],
        'Criticidade': ctx['criticidade'],
        'Decisão':     res['decisao'],
    })

df = pd.DataFrame(linhas)

def colorir_decisao(val):
    cores = {
        'decisao(GO)':         'background-color: #2ecc71; color: white; font-weight: bold',
        'decisao(MONITORAR)':  'background-color: #a8e6a3; color: black; font-weight: bold',
        'decisao(AJUSTAR)':    'background-color: #f39c12; color: white; font-weight: bold',
        'decisao(INVESTIGAR)': 'background-color: #e67e22; color: white; font-weight: bold',
        'decisao(NOGO)':       'background-color: #e74c3c; color: white; font-weight: bold',
        'decisao(INDEFINIDA)': 'background-color: #95a5a6; color: white; font-weight: bold',
    }
    return cores.get(val, '')

display(
    df.style
    .applymap(colorir_decisao, subset=['Decisão'])
    .set_caption('Motor de Inferência com Fatos Reais')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
        {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-size', '13px'), ('padding', '8px 12px')]},
        {'selector': 'td', 'props': [('padding', '8px 12px'), ('font-size', '13px'), ('border-bottom', '1px solid #ddd')]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f5f5f5')]},
    ])
    .hide(axis='index')
)

C:\Users\44057824820\AppData\Local\Temp\ipykernel_7604\3742544650.py:31: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(colorir_decisao, subset=['Decisão'])


ID,Azure (tag),Confiança,Vibração,Criticidade,Decisão
scratch_0,leg_broken,alta,no_defected,alta,decisao(NOGO)
scratch_1,bed_no_stick,alta,no_defected,media,decisao(INDEFINIDA)
scratch_2_10,no_support,alta,no_defected,media,decisao(INDEFINIDA)
scratch_2_9,no_bottom,baixa,no_defected,alta,decisao(INDEFINIDA)


### 🎯 Exercício: ajuste os parâmetros das RULES

Modifique `prioridade` e `condicoes` e observe o efeito no motor.

Perguntas para guiar:
- A Regra A deve ter prioridade maior ou menor que `R04_NOGO_RISCO_ALTO_CRITICO`?
- Se trocar `confianca(baixa)` por `confianca(alta)` na Regra B, o que acontece com P3?
- E se adicionar `criticidade(alta)` como condição extra na Regra B?
- O que acontece se o Azure retornar `no_support` com alta confiança? O motor tem resposta?

---
## 10. Discussão Final

### 🔎 Perguntas

**1. O motor mudou alguma coisa em relação à Aula 4?**


**2. As regras R07 e R09 cobrem apenas `leg_broken`. O que acontece se o Azure retornar `no_support` ou `bed_no_stick` com alta confiança?**

**3. O grounding define que `prob >= 0.80` é confiança alta. Quem deveria definir esse número numa fábrica real? Com base em quê?**

**4. A leitura de vibração foi sorteada do CSV — não corresponde à mesma peça da imagem. Como isso afetaria um sistema em produção?**


### 📋 Para a Próxima Aula — o que vocês vão construir

A Aula 6 mostrou o **mecanismo**. As próximas aulas evoluem o **sistema**.

Três frentes de trabalho:

**1. Ampliar a cobertura das regras**

Hoje só temos regras para `leg_broken`. As demais classes do Azure ficam sem decisão.
Vocês vão criar regras para todas:

| Situação | O que investigar |
|---|---|
| `no_support` + `confianca(alta)` | Falta de suporte confirmada — qual a ação? |
| `bed_no_stick` + `confianca(alta)` | Problema de adesão — para ou ajusta? |
| `no_bottom` + `confianca(alta)` | Fundo ausente — criticidade muda a decisão? |
| `no_defected` + `vibracao(erro)` | Visão ok, vibração não — quem você confia? |

**2. Ampliar o contexto — mais peças, mais variações**

O `contexto_pecas.csv` hoje tem 4 peças. No projeto final vocês vão construir
um contexto real com pelo menos 10 peças — com e sem defeito, variando
criticidade e prazo — para que o motor enfrente situações diversas.

**3. Evoluir para decisão gradual**

Hoje o motor decide entre GO, INVESTIGAR e NOGO.
Na próxima aula incorporamos o que vimos na **Aula 5** — score de risco,
MONITORAR e AJUSTAR — agora com dados reais:
```
prob = 0.91      →  score += 15   (bônus de confiança)
leg_broken       →  score += 45
vibracao(erro)   →  score += 40
──────────────────────────────────
score = 100      →  faixa extremo  →  NOGO
```

A probabilidade real que o Azure devolve deixa de ser só `alta/baixa`
e passa a **alimentar o score diretamente**.

---

### 🗓️ Roteiro até o Projeto Final

| Aula | Foco |
|---|---|
| ✅ Aula 6 | Pipeline real: Azure + SVM → fatos → motor |
| Aula 7 | Novas regras + decisão gradual com dados reais |
| Aula 8 | Integração com modelo de vibração não supervisionada e TIA (Á DEFINIR) |
| Aulas 9–10 | Projeto final: pipeline completo + motor + decisão explicável |